# SDL — generate manifest & publish (form)

A polished, form-driven way to **build a `sdl_manifest.yml` and publish** a Tsunami-HySEA experiment in
one pass (inspired by Demo 4). The technical fields (simulation name, grids, output prefixes, spatial
coverage) are auto-derived; you fill the human metadata in a form where **required fields are marked
`*`** and everything else is optional.

**Flow:** *Setup → Login → (optional) search ids → fill form → (1) Build manifest → review →
(2) Publish*. Use the optional search cells to look up **author / project / organization** ids, and pick
your **code** from the selector — its keywords are filled in automatically.

> ⚠️ *Build* writes `<project>/sdl_manifest.yml` and **overwrites** an existing one (it is for a *new*
> experiment). Requires `parfiles/ bathy/ sim/`. Publishing targets the server you logged into.


## 1. Setup & dependencies

In [ ]:
# --- Setup: load the companion library sdl_lib.py and check dependencies -----
import os, sys, importlib, importlib.util
from pathlib import Path
import ipywidgets as widgets
from IPython.display import display, clear_output

# Locate sdl_lib.py: it lives either in a sibling 01_sdl_publish/ folder (repo
# layout) or directly in the parent folder (server ~/SDL layout). Search both.
_cands = [os.path.join(os.getcwd(), "..", "01_sdl_publish"),
          os.path.join(os.getcwd(), ".."),
          os.getcwd(),
          os.path.expanduser("~/HySEALab/Notebooks/01_sdl_publish"),  # HySEALab menu: shared lib
          os.path.expanduser("~/HySEALab/Notebooks"),                 # shared (next to map viewer)
          os.path.expanduser("~/SDL")]                                # personal working copy
LIB_DIR = next((os.path.abspath(d) for d in _cands
                if os.path.exists(os.path.join(d, "sdl_lib.py"))), None)
if LIB_DIR is None:
    raise FileNotFoundError("sdl_lib.py not found in ../01_sdl_publish, .. or .")
sys.path.insert(0, LIB_DIR)

REQUIRED = {
    "requests": "requests", "requests_toolbelt": "requests-toolbelt",
    "rasterio": "rasterio", "cmocean": "cmocean",
    "matplotlib": "matplotlib", "numpy": "numpy", "tqdm": "tqdm",
    "yaml": "pyyaml", "jsonschema": "jsonschema",
}
_missing = [pip for mod, pip in REQUIRED.items() if importlib.util.find_spec(mod) is None]

_dep_out = widgets.Output()
def _install_missing(_):
    import subprocess
    with _dep_out:
        clear_output()
        print("Installing into the user site:", " ".join(_missing))
        subprocess.run([sys.executable, "-m", "pip", "install", "--user", *_missing])
        print("Done. Restart the kernel and re-run this cell.")
_btn_install = widgets.Button(description="Install missing deps", button_style="warning")
_btn_install.on_click(_install_missing)

if _missing:
    print("Missing dependencies:", ", ".join(_missing))
    display(_btn_install, _dep_out)
else:
    print("All dependencies available.")

import sdl_lib
importlib.reload(sdl_lib)
print("sdl_lib loaded from:", LIB_DIR)

# Shared state for this notebook (login token lives only in memory).
S = {"base_url": "https://sdl-dev.hpc.cineca.it", "token": None, "headers": None}
print("Setup ready.")


## 2. Login

The server you log into is where catalogues are read and where Publish uploads.

In [ ]:
# --- Login: authenticate once; the token is reused by the publisher ----------
w_env = widgets.Dropdown(
    options=[("Development (sdl-dev.hpc.cineca.it)", "https://sdl-dev.hpc.cineca.it"),
             ("Production  (sdl.hpc.cineca.it)", "https://sdl.hpc.cineca.it")],
    value="https://sdl-dev.hpc.cineca.it", description="Server:",
    layout=widgets.Layout(width="440px"))
w_user = widgets.Text(description="Username:", placeholder="your SDL e-mail",
                      layout=widgets.Layout(width="440px"))
w_pass = widgets.Password(description="Password:", layout=widgets.Layout(width="440px"))
w_login = widgets.Button(description="Login", button_style="primary")
out_login = widgets.Output()

def _do_login(_):
    with out_login:
        clear_output()
        S["base_url"] = w_env.value
        u = w_user.value.strip()
        if not u or not w_pass.value:
            print("Enter username and password."); return
        try:
            tok = sdl_lib.get_auth_token(S["base_url"], u, w_pass.value)
        except Exception as e:
            print("Login failed:", e); return
        if not tok:
            print("Login failed: no token returned."); return
        S["token"] = tok
        S["headers"] = sdl_lib.auth_headers(tok)
        print("Login OK - token stored. Server:", S["base_url"])
w_login.on_click(_do_login)
display(widgets.VBox([w_env, w_user, w_pass, w_login, out_login]))


## 3. Catalogue search helpers

Run this once; it enables the optional search cells below.

In [ ]:
# --- Catalogue search helpers (used by the optional search cells below) -------
def _cat_items(kind):
    return (sdl_lib.list_persons(S["base_url"], S["headers"]) if kind == "persons"
            else sdl_lib.list_catalog(S["base_url"], S["headers"], kind))

def _cat_label(kind, o):
    if kind == "persons":
        e = o.get("emails"); em = e[0] if isinstance(e, list) and e else (o.get("email") or "")
        return f"{o.get('givenName','')} {o.get('familyName','')}".strip() + (f"  <{em}>" if em else "")
    if kind == "organizations":
        return o.get("legalName") or o.get("name") or ""
    return o.get("title") or o.get("name") or ""

def _searcher(kind, placeholder):
    q = widgets.Text(description="Filter:", placeholder=placeholder, layout=widgets.Layout(width="480px"))
    b = widgets.Button(description="Search", button_style="info")
    out = widgets.Output()
    def _go(_):
        with out:
            clear_output()
            if not S.get("token"):
                print("Login first (section 2)."); return
            items = _cat_items(kind); ql = q.value.strip().lower()
            rows = [(o.get("id"), _cat_label(kind, o)) for o in items]
            if ql:
                rows = [(i, l) for i, l in rows if ql in l.lower()]
            if not rows:
                print(f"No {kind} match '{q.value}'. Total in catalogue: {len(items)}."); return
            print(f"{kind}: {len(rows)} match(es) - copy the id(s) into the form below.\n")
            for i, l in sorted(rows, key=lambda r: str(r[1]).lower()):
                print(f"  id={str(i):>5}  {l}")
    b.on_click(_go)
    return widgets.VBox([widgets.HBox([q, b]), out])

print("Catalogue search helpers ready (persons / projects / organizations).")


## 4. (Optional) Search **authors** (researchers)

Find person ids to put in *Author ids* (required field). Empty filter lists everyone.

In [ ]:
display(_searcher("persons", "surname / name substring; empty = list all"))


## 5. (Optional) Search **projects**

Find project ids for the optional *Project ids* field.

In [ ]:
display(_searcher("projects", "project title substring; empty = list all"))


## 6. (Optional) Search **institutions** (organizations)

Find organization ids for the optional *Organization ids* field.

In [ ]:
display(_searcher("organizations", "institution name substring; empty = list all"))


## 7. Fill the form → build → publish

Pick the **code** (keywords auto-fill), complete the **required** fields (`*`) and any optional ones,
then **Build manifest** to preview and **Publish to SDL** to upload.

In [ ]:
# --- Form -> generate manifest -> publish (required fields marked *) ----------
import yaml
_reg = getattr(sdl_lib, "CODE_METADATA_REGISTRY", {})
_codes = list(_reg.keys()) or ["tsunami_hysea"]
_ds = {"description_width": "150px"}
_W = widgets.Layout(width="680px")

# REQUIRED
f_proj = widgets.Text(description="Project folder *", style=_ds, layout=_W,
                      value=os.path.abspath(os.path.join(os.getcwd(), "..", "ejemplo_5sims")),
                      placeholder="folder with parfiles/ bathy/ sim/")
f_code = widgets.Dropdown(description="Code *", style=_ds, options=_codes, value=_codes[0],
                          layout=widgets.Layout(width="430px"))
f_cver = widgets.Text(description="Code version *", style=_ds, value="",
                      placeholder="e.g. 1.2.0", layout=widgets.Layout(width="430px"))
f_title = widgets.Text(description="Title *", style=_ds, layout=_W, placeholder="experiment title")
f_auth = widgets.Text(description="Author ids *", style=_ds, layout=_W,
                      placeholder="comma-separated, at least one - e.g. 24,25")
# OPTIONAL / auto-filled
f_keys = widgets.Text(description="Keywords", style=_ds, layout=_W,
                      placeholder="auto-filled from the code; edit freely")
f_desc = widgets.Textarea(description="Description", style=_ds,
                          layout=widgets.Layout(width="680px", height="60px"),
                          placeholder="optional free text")
f_orgs = widgets.Text(description="Organization ids", style=_ds, layout=_W,
                      placeholder="optional, comma-separated - e.g. 41 (UMA)")
f_projs = widgets.Text(description="Project ids", style=_ds, layout=_W,
                       placeholder="optional, comma-separated - e.g. 3 (Geo-INQUIRE)")
f_vis = widgets.Dropdown(description="Visibility", style=_ds, options=["private", "public"], value="private")
f_lic = widgets.Text(description="License", style=_ds, value="cc-by-4.0",
                     placeholder="required only if Visibility = public", layout=widgets.Layout(width="430px"))
f_doi = widgets.Checkbox(value=False, description="requestDOI (optional)")
b_build = widgets.Button(description="1) Build manifest", button_style="primary")
b_pub = widgets.Button(description="2) Publish to SDL", button_style="warning")
out_f = widgets.Output()
ST_F = {"path": None}

# Selecting a code auto-fills its registered keywords (editable afterwards).
def _fill_keys(_=None):
    f_keys.value = ", ".join((_reg.get(f_code.value) or {}).get("keywords", []))
f_code.observe(_fill_keys, names="value"); _fill_keys()

def _ids(s):
    return [int(x) for x in s.replace(";", ",").split(",") if x.strip()]

def _build(_):
    with out_f:
        clear_output()
        proj = f_proj.value.strip()
        if not os.path.isdir(proj):
            print("Project folder * is required and must exist:", proj); return
        if not f_title.value.strip():
            print("Title * is required."); return
        if not f_cver.value.strip():
            print("Code version * is required."); return
        try:
            auth = _ids(f_auth.value); orgs = _ids(f_orgs.value); projs = _ids(f_projs.value)
        except ValueError:
            print("Ids must be integers (e.g. 24,25)."); return
        if not auth:
            print("Author ids * is required (at least one)."); return
        if f_vis.value == "public" and not f_lic.value.strip():
            print("License is required when Visibility = public."); return
        try:
            path = sdl_lib.generate_manifest(proj)
        except Exception as e:
            print("generate failed:", type(e).__name__, e); return
        m = yaml.safe_load(open(path, encoding="utf-8"))
        m["sdl"]["server"] = "dev" if "dev" in (S.get("base_url") or "") else "prod"
        ex = m["experiment"]; ex["mode"] = "new"
        ex["title"] = f_title.value.strip()
        ex["description"] = f_desc.value.strip()
        ex["code"]["name"] = f_code.value
        ex["code"]["version"] = f_cver.value.strip()
        ex["keywords"] = [k.strip() for k in f_keys.value.replace(";", ",").split(",") if k.strip()]
        ex["authors"] = [{"id": i} for i in auth]
        ex["organizations"] = [{"id": i} for i in orgs]
        ex["projects"] = [{"id": i} for i in projs]
        ex["requestDOI"] = bool(f_doi.value)
        m["publication"]["visibility"] = f_vis.value
        m["publication"]["license"] = f_lic.value.strip()
        with open(path, "w", encoding="utf-8") as fh:
            yaml.safe_dump(m, fh, sort_keys=False, allow_unicode=True)
        ST_F["path"] = path
        print("Manifest written:", path)
        print("-" * 70)
        print(open(path, encoding="utf-8").read())
        print("-" * 70)
        print("Review it, then click '2) Publish to SDL'.")
b_build.on_click(_build)

def _pub(_):
    with out_f:
        if not ST_F["path"]:
            print("Build the manifest first (button 1)."); return
        if not S.get("token"):
            print("Login first (section 2)."); return
        print("\nPublishing to", S["base_url"], "...")
        try:
            res = sdl_lib.publish_from_manifest(ST_F["path"], base_url=S["base_url"],
                                                token=S["token"], headers=S["headers"], log=print)
            print("\nDone:", res, "->", S["base_url"] + "/app/")
        except Exception as e:
            print("Publish failed:", type(e).__name__, e)
b_pub.on_click(_pub)

display(widgets.VBox([
    widgets.HTML("<b>Fields marked * are REQUIRED;</b> the rest are optional. "
                 "Keywords are pre-filled from the selected code and can be edited."),
    f_proj, f_code, f_cver, f_title, f_auth,
    widgets.HTML("<hr style='margin:6px 0'><b>Optional</b>"),
    f_keys, f_desc, f_orgs, f_projs, f_vis, f_lic, f_doi,
    widgets.HBox([b_build, b_pub]), out_f]))


## Notes

- **Code selector:** lists only your registered codes (`sdl_lib.CODE_METADATA_REGISTRY`, editable from
  the *Code registry manager*). Selecting one pre-fills its keywords; the publisher also merges them.
- **Required (*)**: project folder, code, code version, title, at least one author id. **Optional:**
  keywords (auto), description, organization ids, project ids, visibility/license, requestDOI.
- **Ids** come from the optional search cells above (or the *Catalogue manager*). License is only needed
  when visibility is *public*.
- Companion `sdl_lib.py` provides `generate_manifest`, `list_persons` / `list_catalog`, and
  `publish_from_manifest`.
